# Neurolucida ASC

The Neurolucida ASCII format (`.asc`), produced by MBF Bioscience's Neurolucida
tracing software, stores morphologies as nested S-expressions with rich
metadata (spines, markers, contours). `braincell` reads it through
{class}`braincell.io.AscReader`.

## Loading

In [ ]:
import braincell

morpho = braincell.Morphology.from_asc("neuron.asc")

As with SWC, you can request a validation report:

In [ ]:
morpho, report = braincell.Morphology.from_asc("neuron.asc", return_report=True)
print(report)   # AscReport: issues found and metadata

## What the reader captures

| Structure | Read as | Details |
|---|---|---|
| Tree marked `(Axon)`, `(Dendrite)` or `(Apical)` | geometry | `axon`, `dendrite` or `apical_dendrite` branches |
| Tree without a type marker | geometry | `custom` branches, with a `topology.untyped_tree` warning naming its line |
| One CellBody contour | geometry | the soma: 21 points along the contour's principal axis, sampled as NEURON 9.0 does, also when the contour is not planar |
| A stack of CellBody contours (layers in z) | geometry | the soma: one point per layer |
| No CellBody contour | geometry | a soma synthesized at the first tree's root point; every tree attaches to it (`topology.synthetic_soma`) |
| Spines (`< ... >` blocks) | metadata | {class}`~braincell.io.AscSpineRecord` entries in `report.metadata.spines`; the parent branch keeps its points |
| Markers, `(Spine ...)` and `(FilledCircle ...)` annotations, colours, comments, names | metadata | fields of {class}`~braincell.io.AscMetadata` |
| Other contours (`"Pia"`, layer outlines), user lines, text labels | metadata | not geometry; their names go to `report.metadata.source_labels` |
| A point that is not four finite numbers `(x y z d)` with `d > 0` | error | `ValueError` naming the line; one trailing label such as `S1` is allowed |
| A point after a branch's child branches | error | `ValueError` naming the line |
| A CellBody contour with fewer than 3 points, collinear points, or a minor axis parallel to z | error | `ValueError` naming the contour's line |
| Two or more separate CellBody groups (several cells) | error | `ValueError`; split the file into one cell per file |
| Unbalanced parentheses, an unterminated string, a file with no geometry | error | `ValueError` |

A contour is the soma when it has a `(CellBody)` marker or is named exactly
`"CellBody"`, `"Cell Body"` or `"Soma"`. A file that is not valid UTF-8 is read
as Latin-1, with a `syntax.encoding_fallback` warning. Warnings go into the
{class}`~braincell.io.AscReport` (a list of {class}`~braincell.io.AscIssue`)
returned with `return_report=True`; they are not Python warnings.

### Differences from NEURON

The geometry matches NEURON 9.0's `Import3d_Neurolucida3` on every input both
accept, with two exceptions:

- A contour with a `(CellBody)` marker but another name (`"Outline"`) is the
  soma here. NEURON goes by the three names only and drops it.
- When there is no CellBody contour, the trees share one synthesized soma here.
  NEURON builds no soma and keeps each tree as a separate root section. The
  `topology.synthetic_soma` warning gives the number of trees and the distance
  from the soma to the farthest root point.

braincell is also stricter. It raises on a point without a diameter, on a
diameter of 0 or less, and on a CellBody contour with fewer than 3 points.
NEURON accepts the first two and drops the contour. braincell also raises on a
file with several cells, which NEURON reads as several root sections. NEURON
before 9.0 can order a single-contour soma from the other end, so compare with
NEURON 9.0 or later.

## Lower-level reader

In [ ]:
from braincell.io import AscReader

reader = AscReader()
morpho = reader.read("neuron.asc")

See {doc}`../apis/io` for the full ASC API. For the simpler, more portable
format, see {doc}`swc`.